# Fine-tuning a pretrained ResNet18 (P >> N)

Genuine dataset-overparam: ImageNet-pretrained ResNet18 (~11M params) fine-tuned on small CIFAR subsets. Test accuracy vs N (P/N), Sven vs AdamW/SGD/Muon.

> Loads the fresh Gram-backend results. Robust to partial data (plots whatever has finished).

In [ ]:
import sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '.')
from style import load_results, average_over_seeds, set_style
from analysis_helpers import add_derived, best_per_method, valid, loss_curve, steps_to_target, method_order
from pathlib import Path
set_style()
PLOT_DIR = Path('plots/finetune'); PLOT_DIR.mkdir(parents=True, exist_ok=True)
def sven_color(m): return 'k' if m=='Sven' else None
def sven_lw(m):    return 2.6 if m=='Sven' else 1.6

In [ ]:
df = add_derived(load_results('exp_finetune_cifar_smallN'))
P = 11_181_642

### Best val accuracy vs training-set size (P/N)

In [ ]:
b=best_per_method(df, by='final_val_loss', extra_group=['n_data'])
fig,ax=plt.subplots(figsize=(6.8,4.4))
for m in method_order(b['method'].unique()):
    s=b[b.method==m].sort_values('n_data')
    ax.plot(s['n_data'], s['final_val_acc'], marker='o', color=sven_color(m), lw=sven_lw(m), label=m)
ax.set_xscale('log'); ax.set_xlabel('N train (P/N = %.0f/N)'%P); ax.set_ylabel('best val accuracy'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(PLOT_DIR/'finetune_acc_vs_N.pdf',bbox_inches='tight'); plt.show()

### Table

In [ ]:
b=best_per_method(df, by='final_val_loss', extra_group=['n_data'])
for nd in sorted(b['n_data'].unique()):
    print(f'== N={nd}  (P/N={P/nd:.0f}) ==')
    for _,r in b[b.n_data==nd].sort_values('final_val_loss').iterrows():
        print(f'  {r.method:8s} val_acc={r.final_val_acc:.3f}  val_loss={r.final_val_loss:.4f}')